# Aula 03 — Programação Orientada a Objetos

Modelar contas e transações com **classes pequenas e testáveis**: `@dataclass`, `property`,
`Enum`, herança com classe abstrata, composição e exceção de domínio.

A regra de negócio fica em `src/opencode_harness/contas.py` e os testes em `tests/test_contas.py`.
O notebook orquestra e mostra o comportamento — o código que precisa de garantia já tem teste.

## 1. Classes e objetos

Uma `Transacao` guarda **o quê**, **quanto**, **que direção** e **quando**. Criar o objeto é
instanciar a classe; cada instância carrega o próprio estado.

In [1]:
from __future__ import annotations

import sys

from opencode_harness import (
    Carteira,
    Conta,
    ContaCorrente,
    ContaPoupanca,
    SaldoInsuficienteError,
    TipoMovimento,
    Transacao,
)

print(f"Python {sys.version.split()[0]}")
print("classes:", [t.__name__ for t in (Transacao, ContaCorrente, ContaPoupanca, Carteira)])

Python 3.14.6
classes: ['Transacao', 'ContaCorrente', 'ContaPoupanca', 'Carteira']


## 2. `@dataclass` imutável e `Enum`

`frozen=True` impede alteração depois da criação, e `Enum` troca strings soltas por um conjunto
fechado de opções. `__post_init__` valida na entrada: objeto inválido não chega a existir.

In [2]:
salario = Transacao("salário de novembro", 3_200.00, TipoMovimento.ENTRADA)
aluguel = Transacao("aluguel", 1_500.00, TipoMovimento.SAIDA)

print(salario)
print("entrada soma:", salario.assinatura)
print("saída subtrai:", aluguel.assinatura)
print("tipo como valor:", salario.tipo.value)

Transacao(descricao='salário de novembro', valor=3200.0, tipo=<TipoMovimento.ENTRADA: 'entrada'>, quando=datetime.date(2026, 11, 11))
entrada soma: 3200.0
saída subtrai: -1500.0
tipo como valor: entrada


Imutabilidade e validação na prática:

In [3]:
from dataclasses import FrozenInstanceError

try:
    salario.valor = 9_999.00
except FrozenInstanceError:
    print("Transacao é imutável: o valor não muda depois de criado")

for valor in (0, -10):
    try:
        Transacao("inválida", valor, TipoMovimento.ENTRADA)
    except ValueError as erro:
        print(f"valor {valor}: {erro}")

Transacao é imutável: o valor não muda depois de criado
valor 0: valor da transação deve ser positivo
valor -10: valor da transação deve ser positivo


## 3. Encapsulamento com `property`

O saldo só muda pelos métodos `depositar` e `sacar`. `property` expõe leitura sem abrir a porta
para escrita, e `historico` devolve uma **tupla** — quem consulta não consegue alterar a lista interna.

In [4]:
corrente = ContaCorrente("Ana", saldo_inicial=1_000.00, limite=500.00)
corrente.depositar(200.00)
corrente.sacar(1_300.00)

print("saldo:", corrente.saldo)
print("disponível (saldo + limite):", corrente.saldo_disponivel)
print("histórico é", type(corrente.historico).__name__, "com", len(corrente.historico), "itens")

try:
    corrente.saldo = 1_000_000.00
except AttributeError:
    print("saldo é somente leitura: a property não tem setter")

saldo: -100.0
disponível (saldo + limite): 400.0
histórico é tuple com 2 itens
saldo é somente leitura: a property não tem setter


## 4. Exceção de domínio

`SaldoInsuficienteError` herda de `ValueError`, então quem já trata erro de valor continua
funcionando — mas o nome deixa claro **qual regra** foi violada. E o saldo não muda quando o saque falha.

In [5]:
antes = corrente.saldo

try:
    corrente.sacar(10_000.00)
except SaldoInsuficienteError as erro:
    print("bloqueado:", erro)
    print("é um ValueError?", isinstance(erro, ValueError))

print("saldo não mudou:", corrente.saldo == antes)

bloqueado: saque de R$ 10000.00 acima do disponível (R$ 400.00)
é um ValueError? True
saldo não mudou: True


## 5. Herança e classe abstrata

`Conta` não pode ser instanciada: ela define o contrato (`tarifa_mensal`) que cada filha cumpre
do seu jeito. O laço abaixo chama o mesmo método e obtém respostas diferentes — isso é polimorfismo.

In [6]:
poupanca = ContaPoupanca("Bia", saldo_inicial=5_000.00, taxa_mensal=0.005)

for conta in (corrente, poupanca):
    print(
        f"{type(conta).__name__:14s} tarifa: R$ {conta.tarifa_mensal():5.2f}"
        f" | saldo: R$ {conta.saldo:8.2f}"
    )

print("poupança também é uma Conta:", isinstance(poupanca, Conta))

try:
    Conta("sem tipo")
except TypeError:
    print("Conta é abstrata: só as filhas implementam tarifa_mensal")

ContaCorrente  tarifa: R$ 12.00 | saldo: R$  -100.00
ContaPoupanca  tarifa: R$  0.00 | saldo: R$  5000.00
poupança também é uma Conta: True
Conta é abstrata: só as filhas implementam tarifa_mensal


## 6. Composição: `Carteira`

Herança diz "é um"; composição diz "tem um". `Carteira` **tem** contas, delega o saldo para elas
e agrega o total — sem herdar nada de `Conta`.

In [7]:
carteira_exemplo = Carteira([corrente, poupanca])
carteira_exemplo.adicionar(ContaCorrente("Ana", saldo_inicial=250.00, limite=100.00))

print(carteira_exemplo)
print("por titular:", carteira_exemplo.por_titular())
print("contas:", [type(conta).__name__ for conta in carteira_exemplo])

Carteira(3 contas, saldo_total=5150.00)
por titular: {'Ana': 150.0, 'Bia': 5000.0}
contas: ['ContaCorrente', 'ContaPoupanca', 'ContaCorrente']


## 7. Mini-projeto: fechamento do mês

Juntando tudo: duas contas da mesma titular, um mês de movimentação e as verificações com `assert`.

In [8]:
conta = ContaCorrente("Ana", saldo_inicial=2_500.00, limite=1_000.00)
poupanca = ContaPoupanca("Ana", saldo_inicial=4_000.00, taxa_mensal=0.005)
carteira = Carteira([conta, poupanca])

conta.depositar(3_200.00, "salário")
conta.sacar(1_500.00, "aluguel")
conta.sacar(420.00, "mercado")
rendimento = poupanca.render()

entradas = sum(t.valor for t in conta.historico if t.tipo is TipoMovimento.ENTRADA)
saidas = sum(t.valor for t in conta.historico if t.tipo is TipoMovimento.SAIDA)

print(f"entradas:  R$ {entradas:8.2f}")
print(f"saídas:    R$ {saidas:8.2f}")
print(f"rendimento da poupança: R$ {rendimento:.2f}")
print(f"saldo total da carteira: R$ {carteira.saldo_total:.2f}")

assert conta.saldo == 3_780.00
assert rendimento == 20.00
assert poupanca.saldo == 4_020.00
assert carteira.saldo_total == 7_800.00
assert conta.tarifa_mensal() == 12.00

try:
    conta.sacar(conta.saldo_disponivel + 0.01)
except SaldoInsuficienteError:
    pass
else:
    raise AssertionError("saque acima do limite deveria falhar")

print("mini-projeto OK")

entradas:  R$  3200.00
saídas:    R$  1920.00
rendimento da poupança: R$ 20.00
saldo total da carteira: R$ 7800.00
mini-projeto OK


## Conclusões

O que treinamos aqui:

- **classe e objeto**: molde e instância, cada uma com seu estado;
- **`@dataclass(frozen=True)`**: menos código repetido e objeto imutável, validado no `__post_init__`;
- **`property`**: leitura pública, escrita controlada pelos métodos;
- **`Enum`**: direção da transação sem string solta;
- **exceção de domínio**: erro com nome próprio, sem quebrar quem já trata `ValueError`;
- **herança + ABC**: contrato na base, regra específica em cada filha;
- **composição**: `Carteira` tem contas e agrega o total;
- **separação**: lógica em `src/`, teste em `tests/`, notebook só demonstra.

> **Dinheiro com `float`:** aqui usamos `float` + `round(..., 2)` para manter o foco em POO. Em
> sistema real, use `Decimal` ou guarde centavos como `int`: `0.1 + 0.2` não é exatamente `0.3`.

**Próximos passos**

1. Módulo 04 — Banco de dados e SQL: persistir essas transações em SQLite e consultar com `SELECT`.
2. Adicionar `extrato()` formatado e `transferir()` entre contas da mesma carteira.
3. Cobrir com teste o caminho feliz **e** os erros de validação (já são 10 em `tests/test_contas.py`).

**Checklist de entrega**

- [x] Notebook executa de ponta a ponta sem erro
- [x] `uv run pytest -q` verde
- [x] `uv run ruff check .` limpo
- [ ] PR aberto e revisão automática lida
- [ ] Merge feito e módulo marcado em `docs/DIO-ROADMAP.md`